# Continuous Contract Rollover Validation
只驗證 TXF `1!` 的單一換月窗口。預設直接跑本窗口，沒有其他 phase。
固定：2026-09-10～2026-09-22、regular、1m、asc、1!/202609/202610。
三組 **完全相同日期範圍**，包含舊契約到期後範圍；不得縮短或換日。

沿用原本四個 Colab Secrets。Run All：mount Drive、官方 SDK setup、登入、三組 read-only requests、diagnostics 與 archive。
既有 canonical document archive 必須存在且 SHA256 相符；不修改 Phase 1 output。
LIVE ROLLOVER API NOT YET TESTED（開發交付狀態）；synthetic tests 不代表實際換月已驗證。


## Environment / Git / Secrets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os, subprocess, sys, tempfile
from google.colab import userdata
REPO_URL = "https://github.com/hh4832/fubon_futures_intraday_data_validation.git"
REPO_DIR = Path('/content/fubon_futures_intraday_data_validation_rollover')
BRANCH = 'feature/continuous-contract-rollover-validation'
# Public clone first; only a private repo needs optional GITHUB_TOKEN.
if not REPO_DIR.exists():
    result = subprocess.run(['git','clone','--branch',BRANCH,REPO_URL,str(REPO_DIR)],capture_output=True,text=True)
    if result.returncode:
        try:
            token = userdata.get('GITHUB_TOKEN')
        except Exception:
            raise RuntimeError('Public clone failed; private repositories need GITHUB_TOKEN') from None
        if not token:
            raise RuntimeError('GITHUB_TOKEN unavailable')
        with tempfile.TemporaryDirectory() as tmp:
            askpass = Path(tmp)/'askpass.py'
            askpass.write_text('#!/usr/bin/env python3\nimport os,sys\nprint("x-access-token" if "Username" in sys.argv[1] else os.environ["COLAB_CLONE_TOKEN"])\n')
            askpass.chmod(0o700)
            env = dict(os.environ,GIT_ASKPASS=str(askpass),GIT_TERMINAL_PROMPT='0',COLAB_CLONE_TOKEN=token)
            retry = subprocess.run(['git','clone','--branch',BRANCH,REPO_URL,str(REPO_DIR)],env=env,capture_output=True)
            del env,token
            if retry.returncode:
                raise RuntimeError('Authenticated clone failed; output suppressed')
remote = subprocess.check_output(['git','remote','get-url','origin'],cwd=REPO_DIR,text=True).strip()
if remote != REPO_URL:
    raise RuntimeError('Repository origin differs from expected URL')
os.chdir(REPO_DIR)
sys.path.insert(0,str(REPO_DIR))
subprocess.run([sys.executable,'-m','pip','install','-r','requirements.txt'],check=True)

actual_branch = subprocess.check_output(['git','branch','--show-current'],cwd=REPO_DIR,text=True).strip()
if actual_branch != BRANCH:
    raise RuntimeError('Use the rollover feature branch checkout; no automatic branch replacement')

In [ ]:
from src.archive import git_info
provenance = git_info(REPO_DIR)
print(provenance)
if provenance['git_status']:
    raise RuntimeError('Use a clean Git checkout; no auto pull or overwrite')

In [ ]:
import platform
print({'python':sys.version,'executable':sys.executable,'platform':platform.platform(),'architecture':platform.machine()})
from src.config import load_config
cfg = load_config()
from src.continuous_rollover import load_rollover_config
cfg = load_rollover_config()
print({k:cfg[k] for k in ["product","session","timeframe","window_from","window_to","continuous_contract","old_contract","new_contract"]})

In [ ]:
from tqdm.auto import tqdm
from scripts.install_fubon_sdk_colab import install
for _ in tqdm(range(1),desc='SDK setup'):
    sdk_info = install(cfg['sdk_version'],cfg['sdk_download_url'],cfg['sdk_official_sha256'])

In [ ]:
import fubon_neo
from fubon_neo.sdk import FubonSDK
print("SDK import PASS")

In [ ]:
from src.fubon_auth import load_secrets, validate_certificate, login
secrets = load_secrets(userdata.get)

In [ ]:
validate_certificate(secrets)

In [ ]:
sdk, auth_info = login(secrets)
del secrets

In [ ]:
try:
    sdk.init_realtime()
    restfutopt = sdk.marketdata.rest_client.futopt
except Exception:
    raise RuntimeError('Market-data initialization failed; SDK exception suppressed') from None
from src.historical_client import read_canonical_documentation
_, documentation_info = read_canonical_documentation(cfg)
print(documentation_info)


## Fixed-window rollover validation
1. 三組 raw responses 分別保存。
2. 保留 per-bar contractMonth，找出 observed transition。
3. OHLC / volume exact comparison；average 單獨列出。
4. 比較前一天、當天、後一天的 additive / ratio adjustment evidence；不調整資料。
5. 列出 reference absence、duplicates、third month、partial-day switch、mismatches。
6. observed volume crossover 只用本窗口 minute-candle sums，不證明 provider 規則。

即使逐根吻合，單一窗口也不能直接授權多年 continuous return 計算。


In [ ]:
from src.continuous_rollover import run_rollover
run_dir, summary = run_rollover(restfutopt, REPO_DIR, sdk_info, auth_info, cfg)
print(summary)


## Diagnostics / archive

In [ ]:
import pandas as pd
for name in ['rollover_contract_coverage','rollover_transition_summary','rollover_field_comparison',
             'rollover_adjustment_test','rollover_volume_comparison','rollover_daily_summary','rollover_anomalies']:
    path=run_dir/'diagnostics'/f'{name}.csv'
    print(name)
    display(pd.read_csv(path))
print('Drive output directory:',str(run_dir.resolve()))


## 結論界線
Observed fact / Documented fact / Inference / Unresolved 分開標示。
契約 transition gap 不等於市場真實報酬。
同源吻合不是 external validation；夜盤與多年 methodology stability 尚未驗證。
沒有任何策略、其他日期取樣、night session 或 higher timeframe request。
